# Experiment: AI Worker 발표·학습용 실험 근거

이 노트북은 저장된 실험 JSON과 발표용 시각자료를 다시 읽어 수치와 도식을 함께 확인한다. identity 후보 검색 수치와 proxy 수치를 섞지 않는다.


## 1. 재현 조건

발표 자료 생성기는 `ai-worker/tools/build_ai_presentation.py`이다. 원본 JSON의 SHA-256과 평가 범위는 `presentation_data.json`에 기록된다.


In [ ]:
from pathlib import Path
import json

repo_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'output' / 'ai-presentation' / 'presentation_data.json').exists())
data = json.loads((repo_root / 'output' / 'ai-presentation' / 'presentation_data.json').read_text(encoding='utf-8'))
headline = data['headline']
identity = data['identity_evidence']
zone = data['zone_evidence']
print('selected:', headline['selected_orchestration'], '/', headline['role'])
print('strict best:', identity['strict_best_method'], identity['strict_best_rank1'], identity['strict_best_recall_at_5'], identity['strict_best_mrr'])
print('improvement: +%.2f pp (x%.1f) from %s' % (headline['rank1_delta_points'], headline['rank1_ratio'], headline['baseline_name']))
print('zone routing proxy:', zone['selection_validation_accuracy'], 'promotion:', zone['promotion_accepted'])

## 2. 시각자료

차트 순서는 발표 순서와 같다. 결론 요약 → 시스템 구조 → 모델 오케스트레이션 → 모델 비교 3종 → proxy 실험 2종 → 근거 상태표.

- 오케스트레이션 도식은 순차 게이트, 병렬 증거 branch, late fusion 경계, 별도 구역 routing, 오프라인 teacher 계층을 각각 다른 표기로 구분한다.
- 버블차트의 원 크기는 Recall@5, 세로축은 strict Rank-1, 가로축은 파라미터 수(로그 축)다.
- Sonnet·구역 차트는 proxy이므로 identity Rank-1과 합산하지 않는다.


In [ ]:
from IPython.display import SVG, display

chart_names = ['headline_summary.svg', 'architecture_pipeline.svg', 'model_orchestration.svg', 'identity_strict_ranked.svg', 'model_evolution.svg', 'identity_model_bubble.svg', 'sonnet_ablation.svg', 'zone_proxy_validation.svg', 'evidence_status.svg']
for chart_name in chart_names:
    display(SVG(filename=str(repo_root / 'output' / 'ai-presentation' / chart_name)))

## 3. 발표 해석

현재 운영 선택은 strict 비교군에서 가장 좋은 Top-K 후보 검색 오케스트레이션(`hybrid-solider-clip-v1`)이다. 초기 baseline 대비 +30.53%p, 약 2.8배 개선했으며 관리자에게 시간·bbox·crop 증거를 반환한다. 자동 신원확정은 별도 후속 sealed gate로 관리한다.
